# Reusable Cleaning Pipeline

**Level:** Advanced

## Project description

You receive messy CSV exports every week. Instead of manually cleaning each
file, build reusable Pandas functions that normalize columns, clean labels,
validate required fields, and calculate revenue.

## Skills practiced

- Writing reusable cleaning functions
- Using pipe()
- Validating data quality
- Keeping transformations readable


In [ ]:
import pandas as pd
import numpy as np
import secrets

practice_run_name = f"cleaning-pipeline-{secrets.token_hex(3)}"
print(f"Practice run: {practice_run_name}")

def make_messy_sales_export(row_count=20, seed=42):
    """Create a synthetic messy CSV-style export for cleaning practice."""
    rng = np.random.default_rng(seed)
    raw = pd.DataFrame({
        " Order ID ": range(1, row_count + 1),
        "Region ": rng.choice([" west", "EAST", "South ", "north", "WEST "], size=row_count),
        "Units Sold": rng.integers(1, 12, size=row_count).astype(float),
        "Unit Price": rng.choice([45, 80, 120, 150, 220], size=row_count),
    })
    raw.loc[rng.choice(raw.index, size=max(1, row_count // 8), replace=False), "Units Sold"] = np.nan
    return raw

raw = make_messy_sales_export(row_count=20, seed=42)
raw


## Step 1: Create a function to normalize column names


In [ ]:
def normalize_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with clean snake_case column names."""
    cleaned = data.copy()
    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return cleaned


## Step 2: Create a function to clean text labels


In [ ]:
def clean_region_labels(data: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with consistent region labels."""
    cleaned = data.copy()
    cleaned["region"] = cleaned["region"].str.strip().str.title()
    return cleaned


## Step 3: Create a validation function


In [ ]:
def validate_required_columns(data: pd.DataFrame, required_columns: list[str]) -> pd.DataFrame:
    """Raise a clear error when required columns are missing."""
    missing_columns = set(required_columns) - set(data.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    return data


## Step 4: Create a function for business metrics


In [ ]:
def add_revenue(data: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with revenue calculated from units and price."""
    cleaned = data.copy()
    cleaned["units_sold"] = cleaned["units_sold"].fillna(0)
    cleaned["revenue"] = cleaned["units_sold"] * cleaned["unit_price"]
    return cleaned


## Step 5: Combine the functions with pipe()


In [ ]:
required = ["order_id", "region", "units_sold", "unit_price"]

clean = (
    raw
    .pipe(normalize_columns)
    .pipe(validate_required_columns, required_columns=required)
    .pipe(clean_region_labels)
    .pipe(add_revenue)
)

clean


## Step 6: Create a validation report


In [ ]:
validation_report = pd.DataFrame({
    "check": [
        "row_count",
        "missing_units_sold_after_cleaning",
        "missing_region_after_cleaning",
        "total_revenue",
    ],
    "value": [
        len(clean),
        clean["units_sold"].isna().sum(),
        clean["region"].isna().sum(),
        clean["revenue"].sum(),
    ],
})

validation_report


## Final project notes

- Which function is most reusable:
- Which validation check protects the analysis most:
- One additional function you would add:
- One reason pipe() improves readability:


## Practice extension: Generate your own dataset

Change `row_count` and `seed` to create a new messy export. Set `seed=None`
for a randomized dataset each run. You can also add more messy region labels
inside the function to test your cleaning pipeline.


In [ ]:
my_raw = make_messy_sales_export(row_count=50, seed=12)
my_raw.head()
